# Token-weighted gradient accumulation with lax.scan

When a batch does not fit in memory, split it into $M$ micro-batches, accumulate gradients and take one optimizer step. The accumulated gradient must equal the gradient of the **full** batch, even when micro-batches contain different numbers of real (unpadded) examples.

Model: a one-hidden-layer network with `params = {'w1': (D, H), 'b1': (H,), 'w2': (H,), 'b2': ()}`:

$$f(x) = \tanh(x W_1 + b_1)\, w_2 + b_2$$

Batch: `x` `(M, b, D)`, `y` `(M, b)` and `mask` `(M, b)` of 0/1 (1 = real example). The full-batch loss is the mean squared error over **real** examples only:

$$L = \frac{\sum_{m, i} \text{mask}_{m,i}\,\bigl(f(x_{m,i}) - y_{m,i}\bigr)^2}{\max\bigl(1,\ \sum_{m,i} \text{mask}_{m,i}\bigr)}$$

Write `accumulate_and_step(params, x, y, mask, lr)`:

1. Loop over micro-batches with `jax.lax.scan`, computing each micro-batch's **summed** masked loss and its gradient with `jax.value_and_grad`, and carrying the running gradient sum, loss sum and example count.
2. Divide by $\max(1, \text{count})$ to get $L$ and $\nabla L$.
3. Take one SGD step: $p \leftarrow p - \eta \nabla L$.

Return `(new_params, loss, grads)`: two dicts with the structure of `params` and a 0-d loss. float32. If no example is real, the loss is 0, the gradients are zero and the parameters are unchanged.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def accumulate_and_step(params, x, y, mask, lr):
    """x (M, b, D), y/mask (M, b). Return (new_params, loss, grads)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["w1", {"__nd__": [[0.0]], "dtype": "float64"}], ["b1", {"__nd__": [0.0], "dtype": "float64"}], ["w2", {"__nd__": [0.0], "dtype": "float64"}], ["b2", {"__nd__": 0.0, "dtype": "float64"}]]}, {"__nd__": [[[0.0]], [[0.0]]], "dtype": "float64"}, {"__nd__": [[1.0], [3.0]], "dtype": "float64"}, {"__nd__": [[1.0], [1.0]], "dtype": "float64"}, 0.5], "e": {"__tup__": [{"__dict__": [["b1", {"__nd__": [0.0], "dtype": "float32"}], ["b2", {"__nd__": 2.0, "dtype": "float32"}], ["w1", {"__nd__": [[0.0]], "dtype": "float32"}], ["w2", {"__nd__": [0.0], "dtype": "float32"}]]}, {"__nd__": 5.0, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [0.0], "dtype": "float32"}], ["b2", {"__nd__": -4.0, "dtype": "float32"}], ["w1", {"__nd__": [[0.0]], "dtype": "float32"}], ["w2", {"__nd__": [0.0], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[0.0]], "dtype": "float64"}], ["b1", {"__nd__": [0.0], "dtype": "float64"}], ["w2", {"__nd__": [0.0], "dtype": "float64"}], ["b2", {"__nd__": 0.0, "dtype": "float64"}]]}, {"__nd__": [[[0.0], [0.0]], [[0.0], [0.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0], [4.0, 0.0]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0], [1.0, 0.0]], "dtype": "float64"}, 0.5], "e": {"__tup__": [{"__dict__": [["b1", {"__nd__": [0.0], "dtype": "float32"}], ["b2", {"__nd__": 2.0, "dtype": "float32"}], ["w1", {"__nd__": [[0.0]], "dtype": "float32"}], ["w2", {"__nd__": [0.0], "dtype": "float32"}]]}, {"__nd__": 6.0, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [0.0], "dtype": "float32"}], ["b2", {"__nd__": -4.0, "dtype": "float32"}], ["w1", {"__nd__": [[0.0]], "dtype": "float32"}], ["w2", {"__nd__": [0.0], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[-0.3700247693240037, 0.994078692032213, 0.41585748064632017, -0.6181637087164065], [0.6721408015696635, -1.450049505885249, 0.5934846357693563, -0.5618297235440954], [0.631453096950567, 0.4390906518766011, -0.76724880591175, 0.5283687488069336]], "dtype": "float64"}], ["b1", {"__nd__": [0.03419276735141769, -0.06461430903975195, 0.20011505552131661, 0.080195993578643], "dtype": "float64"}], ["w2", {"__nd__": [-1.1820652577744168, -0.9879819159784637, 0.3168316854272728, 0.3120442520839395], "dtype": "float64"}], ["b2", {"__nd__": 0.1, "dtype": "float64"}]]}, {"__nd__": [[[-0.7268560483413962, 1.2158778403220516, 0.09701711609506954], [-0.8626833404289537, -0.3745977101286351, -0.17879339436774003], [-1.5533105934432883, -0.8322121471805926, -1.809912000528707], [-0.29355617314731525, 0.612639535701211, -1.2858506769982023]], [[0.32208625997571905, -1.1826669740084903, -0.16362570180203553], [-0.5161660210815051, -1.212183196659453, -0.5490984987838816], [-1.2254692506871665, 0.2935107219230474, 0.04617552533742384], [1.199963415268209, -1.463253289545785, 1.0927759442700036]], [[-0.07875734069918894, 0.4053617273724596, -0.18846678911157366], [1.5469678237589886, 1.3860177942471807, 1.0685069539798961], [0.2594567645115784, -2.3378152457678594, 0.5563281397304826], [-0.03275917922956447, -0.1339695891109679, 0.43455966825756104]]], "dtype": "float64"}, {"__nd__": [[1.1934595796502443, 0.26727027849946766, -0.4664258685088374, -0.7777408134163687], [0.37007088635135027, 1.0985695969740756, 0.5479772180217485, 0.45783840692823724], [-0.45099467108319397, -1.2358991220763367, 0.9460468883208342, 0.606364239571162]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0, 1.0, 1.0], [1.0, 0.0, 0.0, 0.0], [1.0, 1.0, 0.0, 0.0]], "dtype": "float64"}, 0.1], "e": {"__tup__": [{"__dict__": [["b1", {"__nd__": [0.17445167899131775, 0.0007544532418251038, 0.17281009256839752, 0.04906946420669556], "dtype": "float32"}], ["b2", {"__nd__": -0.056221164762973785, "dtype": "float32"}], ["w1", {"__nd__": [[-0.4393824338912964, 0.9673910140991211, 0.44349032640457153, -0.5929011702537537], [0.6966197490692139, -1.4447613954544067, 0.6080575585365295, -0.5510349869728088], [0.480866938829422, 0.3984408974647522, -0.7266655564308167, 0.5758880972862244]], "dtype": "float32"}], ["w2", {"__nd__": [-1.105444312095642, -0.860442578792572, 0.2201906442642212, 0.3241633176803589], "dtype": "float32"}]]}, {"__nd__": 2.0244314670562744, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [-1.4025890827178955, -0.6536876559257507, 0.27304959297180176, 0.3112652897834778], "dtype": "float32"}], ["b2", {"__nd__": 1.5622116327285767, "dtype": "float32"}], ["w1", {"__nd__": [[0.6935765743255615, 0.266876757144928, -0.2763283848762512, -0.2526252865791321], [-0.24478945136070251, -0.05288180336356163, -0.14572900533676147, -0.107947438955307], [1.505861520767212, 0.40649735927581787, -0.40583255887031555, -0.47519320249557495]], "dtype": "float32"}], ["w2", {"__nd__": [-0.766208827495575, -1.2753932476043701, 0.9664103984832764, -0.12119067460298538], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[-1.8276169687063608, -0.7171863622697003, 1.0013699991426948, 1.067796620110215], [-0.6408771819729441, 2.2534637172518237, 1.4017287129961191, 0.9553113236697341], [-0.12767854026500344, -1.391240059944376, 0.8105084042323603, 1.778508002242938]], "dtype": "float64"}], ["b1", {"__nd__": [0.09139625981430416, -0.10778786303832927, 0.11387123535592691, -0.06649229710261388], "dtype": "float64"}], ["w2", {"__nd__": [-2.1980579086793957, 0.32018207972887547, -2.7102773898231143, 0.11305607440390693], "dtype": "float64"}], ["b2", {"__nd__": 0.1, "dtype": "float64"}]]}, {"__nd__": [[[1.0120029078495834, 1.0384322735519498, 2.7208898947966156], [0.6697626143872923, -0.983324083153372, 0.6155073816834921], [0.6791881003528312, -0.582582908162085, -1.7499441556183644], [-0.37369980305301886, -0.6943752303385733, -0.39661106511801364], [-0.18788385714653216, 0.7539269999294069, 0.595614380456181], [1.4259463493646405, -0.5251567356645593, 0.9050846134363121], [-1.7442855042745486, 0.025639894851118006, -1.5849429098341856], [-0.8844839369702756, 1.0437040273566507, -0.9513410741803076]]], "dtype": "float64"}, {"__nd__": [[1.4398326230795224, 1.7167610175581112, 0.5370325642889875, -0.2993926016135853, -0.9823975723654975, -0.8369831716055218, -0.07050628737325618, -0.991666701051195]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]], "dtype": "float64"}, 0.05], "e": {"__tup__": [{"__dict__": [["b1", {"__nd__": [0.1285092830657959, -0.11701083928346634, 0.11195778846740723, -0.0660477951169014], "dtype": "float32"}], ["b2", {"__nd__": 0.08078867197036743, "dtype": "float32"}], ["w1", {"__nd__": [[-1.786237359046936, -0.7231023907661438, 1.0162742137908936, 1.0679433345794678], [-0.6920022964477539, 2.2614951133728027, 1.386839747428894, 0.9552492499351501], [-0.2596086263656616, -1.3720391988754272, 0.7699557542800903, 1.778891921043396]], "dtype": "float32"}], ["w2", {"__nd__": [-2.218331813812256, 0.28052258491516113, -2.627915859222412, 0.20073202252388], "dtype": "float32"}]]}, {"__nd__": 2.0419857501983643, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [-0.7422603964805603, 0.18445949256420135, 0.038268789649009705, -0.008890070952475071], "dtype": "float32"}], ["b2", {"__nd__": 0.38422662019729614, "dtype": "float32"}], ["w1", {"__nd__": [[-0.8275914788246155, 0.11832021921873093, -0.29808565974235535, -0.002935755765065551], [1.0225017070770264, -0.16062551736831665, 0.29778018593788147, 0.0012414753437042236], [2.6386020183563232, -0.3840162754058838, 0.8110535740852356, -0.007678594905883074]], "dtype": "float32"}], ["w2", {"__nd__": [0.40547648072242737, 0.7931900024414062, -1.647230863571167, -1.75351881980896], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[-0.09119582375321082, -0.7789248565889882, 0.614616364712907], [-1.8329609623394012, 0.4827103224233646, 0.9553006455413606]], "dtype": "float64"}], ["b1", {"__nd__": [-0.1003502386487766, 0.14706714799706996, 0.15322585700923963], "dtype": "float64"}], ["w2", {"__nd__": [0.6896922290098128, -1.4922293591919251, 0.27029447560208264], "dtype": "float64"}], ["b2", {"__nd__": 0.1, "dtype": "float64"}]]}, {"__nd__": [[[-1.610053327278038, -0.45567441576570045], [-1.259744326968461, 0.15661282120767445]], [[-0.7461720017755872, 0.3372936211780487], [0.9207492830391933, -0.7050689013558965]], [[0.5404740197098808, -0.41066557864772785], [-0.22189043369406294, -0.3046692739500764]], [[-0.7505086753256622, -0.569731720106415], [0.909432261586648, 1.0412949808366119]]], "dtype": "float64"}, {"__nd__": [[-0.037517213978804774, 0.8326123820893233], [0.988571802286366, 0.6496959224991548], [-2.108025242052095, -0.08377981124548971], [1.2613272768472668, 0.3064069628215977]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, 0.1], "e": {"__tup__": [{"__dict__": [["b1", {"__nd__": [-0.10035023838281631, 0.1470671445131302, 0.1532258540391922], "dtype": "float32"}], ["b2", {"__nd__": 0.10000000149011612, "dtype": "float32"}], ["w1", {"__nd__": [[-0.09119582176208496, -0.7789248824119568, 0.6146163940429688], [-1.8329609632492065, 0.4827103316783905, 0.9553006291389465]], "dtype": "float32"}], ["w2", {"__nd__": [0.6896921992301941, -1.4922293424606323, 0.2702944874763489], "dtype": "float32"}]]}, {"__nd__": 0.0, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [0.0, 0.0, 0.0], "dtype": "float32"}], ["b2", {"__nd__": 0.0, "dtype": "float32"}], ["w1", {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float32"}], ["w2", {"__nd__": [0.0, 0.0, 0.0], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[-0.10202116695196616, -0.3555801249674583, 0.2772256947675629], [-0.3695633796731854, 0.8430965232340522, -0.09515818545886297]], "dtype": "float64"}], ["b1", {"__nd__": [-0.0008857722170152806, 0.04793161315276738, 0.12191124306492473], "dtype": "float64"}], ["w2", {"__nd__": [-0.1545236341775081, 0.44624109503052756, -0.2786514922336978], "dtype": "float64"}], ["b2", {"__nd__": 0.1, "dtype": "float64"}]]}, {"__nd__": [[[-1.2048978591667578, -0.0043195789621606395], [0.5400668305395082, 0.3550315605587094], [-0.09125748629317389, -0.25960763555127403]], [[-1.228722650891579, -0.13175756649900683], [1.5802089635861838, -0.23148591132731963], [-0.04017642290669308, 1.3850074120729572]], [[0.11485780736656534, 0.2501009411106417], [-1.1519781068052937, 0.8145022138046487], [1.3756111090409364, -0.06070320522272527]], [[1.0465839303299564, 0.9706155369930208], [-2.7320888529080154, 1.341553800412967], [-1.4025453618391515, 0.4285116325366789]]], "dtype": "float64"}, {"__nd__": [[0.45118744207035255, -0.23717664400668687, -0.36100151779099887], [-0.3822354159421008, -0.4725777622936688, -0.45432698699715857], [-0.7257642741349587, -0.09754803422832849, 0.09922081079448515], [-0.20524400832939296, 0.30248038981227615, -0.8270826339385826]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0], [1.0, 1.0, 1.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, 0.2], "e": {"__tup__": [{"__dict__": [["b1", {"__nd__": [0.033827804028987885, -0.005846746265888214, 0.18849200010299683], "dtype": "float32"}], ["b2", {"__nd__": -0.1511038839817047, "dtype": "float32"}], ["w1", {"__nd__": [[-0.12229487299919128, -0.3188488185405731, 0.23955273628234863], [-0.3465554118156433, 0.8214581608772278, -0.04539377987384796]], "dtype": "float32"}], ["w2", {"__nd__": [-0.10536634176969528, 0.29317474365234375, -0.2537524402141571], "dtype": "float32"}]]}, {"__nd__": 0.48445814847946167, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [-0.17356787621974945, 0.2688917815685272, -0.33290377259254456], "dtype": "float32"}], ["b2", {"__nd__": 1.2555193901062012, "dtype": "float32"}], ["w1", {"__nd__": [[0.10136851668357849, -0.1836564689874649, 0.18836486339569092], [-0.11503972113132477, 0.10819172859191895, -0.24882200360298157]], "dtype": "float32"}], ["w2", {"__nd__": [-0.24578648805618286, 0.7653316259384155, -0.12449534237384796], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[0.17917095452913478, 0.803171293872631, 1.5616438919919484, -1.2695618988576793, 0.7493720887417908, 0.26173153392908344], [-1.9592711803164862, -0.6300632033377491, -0.998167495682167, 0.5131746383243355, 1.469598358921532, 1.2522300603131846], [0.2406634129973143, 1.372960085071419, 0.3853247870555766, 0.2198943732316332, -1.5154759085276162, -0.7237181293545787], [-0.3698002735752544, -0.7703575038236293, -0.9865668206357533, -0.4763820791683015, 0.9921067293966392, 0.450511424123131], [0.2644912002332324, -0.3595110420246815, -0.45281419761152275, -0.03178921364541607, -0.7424539510664404, 0.16588024620339867]], "dtype": "float64"}], ["b1", {"__nd__": [-0.014613555835249351, -0.03054947299819953, 0.015532685110102827, -0.003453289990463914, 0.02613859664731479, -0.09491704720893968], "dtype": "float64"}], ["w2", {"__nd__": [-1.0065097424225828, -0.8425278110425919, -0.9928058770249327, 1.2189296631466526, -0.8859475010005012, 0.3192803873431583], "dtype": "float64"}], ["b2", {"__nd__": 0.1, "dtype": "float64"}]]}, {"__nd__": [[[0.6503830026210222, 1.3706764216862921, -0.24502866843565574, -0.5919913748407589, 0.13388830067418947], [1.7241540483170323, -0.30822260173574695, 0.8624104371778697, 0.9104286113803851, -1.179665881057688], [1.0860038366865028, 0.6899465238170748, -1.5763482294118574, 0.9174905664507101, 1.205213189190142], [0.05263439753610003, 0.25773228069061094, 0.6857852815719516, 0.3121580304102642, -0.8688257274524804], [-1.1842598595360598, -0.7545559869654416, -0.7361478488431996, 0.5931713671370464, -1.2746643808299392]], [[-0.8590210265997998, -0.8847096876545048, 1.2063000757260913, 1.6486905987415374, 0.42171705596610076], [0.7748213697939366, 0.630827040375287, -0.947416846563734, -0.6752399789996373, 0.445160838907153], [0.0998338710751835, -2.0980686198780236, -0.9768305033289162, -0.07920221134624784, -1.4615711273974608], [0.5544266704687185, 0.5927782788770725, 1.2248685484379371, -1.1524605275994995, -0.27532101090541766], [0.7684370414319148, -0.9213734138261886, -0.006152239919871784, -0.7299480466185145, -0.07779458145504267]], [[-1.2884073760314712, -0.04086549310439323, 1.4751581162240663, 1.0870059793263414, -0.6765217760154042], [-0.18792969974127363, -1.1194924612261032, 0.932054415985456, 0.18645192562135374, 1.011012528505769], [-0.05888402515675665, -0.7134693559128369, 0.6863558183775317, -1.2954389500079107, -0.2591675165031774], [1.0500023434738284, -0.7025382676860087, -2.8183852425308915, -1.10930497728051, 1.76183092338787], [-0.43679673996540225, -1.2878785645286641, -0.5825988352105465, 1.2878207265381187, -0.5538112571470052]], [[-1.1921441858948567, 1.7054199313599003, 0.00630926107988479, 0.7063386638993926, -1.5876161892395892], [1.3449865714055358, 0.12403396389163142, 1.5147979645454663, 1.1352788353660366, -0.0407759379232103], [0.13184813208264343, -0.0013922646754700666, 2.881549017049888, 0.8580815562217632, 1.3647232882995155], [0.7621323306745219, -1.8741531749729174, -1.2698136241417421, 1.5117642068321602, 0.21206854658679208], [-0.6256098325928486, 0.5869348282040326, 0.21350540227816345, 2.2453900222707515, -0.5807538723642469]], [[-0.5951892457781653, 0.3142407875071827, 1.1199696820616825, 0.10369087555845481, 0.7379708935893854], [-0.07897618628511432, -0.07107262342501088, -0.16989241729338514, -0.37414541695485426, 0.9180055322114226], [-0.4219602142925481, 1.0826370485045531, -1.574505794151079, -2.1605383678275802, 2.708626430786477], [-1.1961641895452593, 0.9012508213595188, 0.17870433063564153, 0.9602720236929143, 2.374707223225493], [0.22802966341168437, 0.8235820995800269, -0.33249622033308446, 0.6343752634967678, -1.3081236769473876]], [[1.451825501664888, 0.09531432191999144, -0.9579705191061534, -0.11963702794220955, -0.0848766289858287], [1.3413810891899067, -1.487106747256442, -0.8456001893081201, 0.7157757723753274, -0.49129041492006525], [0.005389414106907368, 0.5427845517130234, -0.8848886886658102, 0.369087576914728, -1.5969999733161084], [-1.4035927785184161, 0.14860041478841818, -0.8432129273244168, -0.882676442342136, -0.9445027769948401], [0.057010613101934414, -0.2705797984142156, 0.022160989081963373, 0.5984236712901841, 1.8775349639970826]]], "dtype": "float64"}, {"__nd__": [[-0.9673589567356546, 3.730742628100572, -0.8715449420355347, -2.993635324589693, 1.138144082876397], [-0.9372716935258963, 5.0154904476621756, -2.7838274003734718, 1.1118365991077401, 2.3949486315128223], [1.7090003912499014, 5.787467745908706, -2.9665611344175424, 0.3551332818706337, 0.6375796038412926], [4.4924053681125695, -4.19549745959816, -4.6596138029053344, -0.8982230838630196, 1.2182320861709282], [-1.4852098885808607, -1.727759836869256, -0.7681394506764839, 1.3547184727145343, 2.867496892756458], [-1.2731575684557037, -3.157090710895421, 1.6400560941717155, -2.149145865682672, 5.61430937252951]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 1.0, 0.0], [1.0, 1.0, 1.0, 0.0, 1.0], [0.0, 1.0, 0.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0, 1.0]], "dtype": "float64"}, 0.01], "e": {"__tup__": [{"__dict__": [["b1", {"__nd__": [-0.020561985671520233, -0.03399783745408058, 0.01606818474829197, -0.00045837066136300564, 0.025674816220998764, -0.09095121920108795], "dtype": "float32"}], ["b2", {"__nd__": 0.12181444466114044, "dtype": "float32"}], ["w1", {"__nd__": [[0.1628865748643875, 0.8000490069389343, 1.5641919374465942, -1.2679601907730103, 0.7419379949569702, 0.2678435444831848], [-1.954193115234375, -0.6222967505455017, -0.9856038689613342, 0.4910530149936676, 1.4758082628250122, 1.2489625215530396], [0.24735328555107117, 1.3725305795669556, 0.39458951354026794, 0.21005475521087646, -1.5130035877227783, -0.7242912650108337], [-0.3746923804283142, -0.7743614912033081, -0.9865642786026001, -0.4788115620613098, 0.9941440224647522, 0.45265379548072815], [0.27580082416534424, -0.3626103401184082, -0.4552554488182068, -0.023406051099300385, -0.7358259558677673, 0.16488158702850342]], "dtype": "float32"}], ["w2", {"__nd__": [-0.9752151370048523, -0.8266846537590027, -0.9655552506446838, 1.1870161294937134, -0.8822294473648071, 0.3039405941963196], "dtype": "float32"}]]}, {"__nd__": 15.331216812133789, "dtype": "float32"}, {"__dict__": [["b1", {"__nd__": [0.5948429703712463, 0.34483638405799866, -0.053549911826848984, -0.2994919419288635, 0.04637794569134712, -0.39658263325691223], "dtype": "float32"}], ["b2", {"__nd__": -2.1814444065093994, "dtype": "float32"}], ["w1", {"__nd__": [[1.6284371614456177, 0.3122289478778839, -0.25480860471725464, -0.16017316281795502, 0.7434054613113403, -0.6112005710601807], [-0.5078052282333374, -0.7766420245170593, -1.2563636302947998, 2.212162733078003, -0.6209807395935059, 0.32674896717071533], [-0.6689873933792114, 0.04294929653406143, -0.9264737963676453, 0.9839618802070618, -0.24722835421562195, 0.05731520801782608], [0.4892125725746155, 0.40040266513824463, -0.0002533912775106728, 0.24294939637184143, -0.20373135805130005, -0.2142382115125656], [-1.1309635639190674, 0.3099285066127777, 0.24412541091442108, -0.8383163213729858, -0.6627951860427856, 0.09986665099859238]], "dtype": "float32"}], ["w2", {"__nd__": [-3.1294660568237305, -1.5843125581741333, -2.725064754486084, 3.1913509368896484, -0.371805876493454, 1.533979058265686], "dtype": "float32"}]]}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(accumulate_and_step, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")